# 推理优化

LLM的推理包括两个阶段。预填充阶段并行地处理你的输入提示词，瓶颈在算力。解码阶段每次吐出一个词元，瓶颈在显存。每个优化目标都是上面之一或者两者。

## 问题描述

单用户的吞吐在50词元每秒，现在有100个用户并发，吞吐量就可能降至3词元每秒。

模型没变，变的只是你的调度器工作不同。原生的推理阶段可能浪费90%的GPU算力。

## 基本概念

### 预填充 VS 解码

每个LLM推理都要求这两个独立的阶段。

**预填充**处理所有的输入提示词。所有的词元都是已知的，所以注意力可以并行地在整个序列上进行计算。这是一次大的矩阵乘法，所以瓶颈在算力。

**解码**每次吐出一个词元。权重矩阵和预填充阶段一样，但是它只乘以一个单独的向量，所以计算量比上个阶段小得多，瓶颈在加载下一批权重到显存中。

算术密度，或者ops::byte ratio 用来衡量这个指标，它衡量了加载到显存中的每字节对应的计算量大小：
```
ops::byte ratio = FLOPS per token / bytes read from memory
```

比如预填充一个长度为4096的词元序列，对于每个加载上来的权重，你需要执行4096次乘法和加法，比率很高。而解码时，加载上来的权重之需要执行1次乘法和加法，比率很低。

基本的洞察就是，解码阶段每次加载所有的权重，但只吐出一个词元。所有的优化策略都是要么降低你读的内容，或者每次读完后想办法增加吞吐量。

### KV缓存

略。。

### 连续批处理

静态批处理在同批次N个请求都完成后，在开始处理下一个批次。单个批次内，先完成的短序列请求会空转知道长序列请求执行完。

连续批处理在请求完成后立即把新的请求插入批处理，避免空转。

这个改进取决于输出的长度变化有多大。如果输出长度基本一致，那么连续批处理的性能与静态批处理相当。

### 分页注意力

核心是不要求KV缓存连续，采用分页+虚拟地址索引的方式管理。
所以对于相同的前缀，可以直接拷贝和复用，比如agent的系统提示词。

### 投机解码

一次吐出多个词元，然后一次前向过程判断接受多少。加速效果取决于接受率的大小。主要方法有：
|方法|草案词元生成|接受率|开销|
|---|---|---|---|
|草案模型|单独的小模型|70-85%|小模型显存|
|EAGLE|额外的输出头，同时预测多个|75-90%|～1% extra parameters|
|N-gram lookup|通过N-gram 匹配|40～60%|忽略不计|

### 前缀缓存

很多请求共享相同的前缀。比如agent系统提示词，RAG系统的上下文块等。
前缀缓存将这些通用前缀的KV Cache缓存下来，在请求需要时做复用。

### 推理引擎

|引擎|关键改进|适合|
|---|---|---|
|vLLM|分页注意力，连续批处理|一般任务，兼容性最好|
|SGLang|根基注意力（前缀缓存），结构化生成|多轮对话，约束解码|
|TensorRT-LLM|NVDIA核融合，FP8量化|NVIDIA硬件单GPU最大化|

核.融合: 指的是GPU核把注意力、线形还有激活流程放在同一个核内计算。

### Ops::Byte 碎碎念

过低，使用量化、增大批等手段加大Ops...

过高，还是得突破Ops瓶颈...

# 动手编码

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class KVCache(nn.Module):
    def __init__(self, num_layers, num_heads, head_dim, max_seq_len, dtype=torch.float16):
        super().__init__()
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        self.dtype = dtype

        self.k_cache = torch.zeros(
            (num_layers, num_heads, max_seq_len, head_dim), dtype=self.dtype
        )
        self.v_cache = torch.zeros(
            (num_layers, num_heads, max_seq_len, head_dim), dtype=self.dtype
        )
        self.seq_len = 0

    def update(self, layer_idx, new_keys, new_values):
        num_new = new_keys.shape[1]
        end = self.seq_len + num_new
        self.k_cache[layer_idx, :, self.seq_len:end, :] = new_keys
        self.v_cache[layer_idx, :, self.seq_len:end, :] = new_values
        return {
            self.k_cache[layer_idx, :, :end, :],
            self.v_cache[layer_idx, :, :end, :],
        }

    def advance(self, num_tokens):
        self.seq_len += num_tokens

    def memory_bytes(self):
        return self.k_cache.nbytes + self.v_cache.nbytes

    def used_bytes(self):
        per_token = 2 * self.num_layers * self.num_heads * self.head_dim * torch.float16.itemsize
        return per_token * self.seq_len

class MHA(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, kv_cache=None, layer_idx=0):
        q = self.q_proj(x)
        q = q.view(q.shape[0], -1, self.num_heads, self.head_dim)
        q = q.transpose(1, 2)

        k = self.k_proj(x)
        k = k.view(k.shape[0], -1, self.num_heads, self.head_dim)
        k = k.transpose(1, 2)

        v = self.v_proj(x)
        v = v.view(v.shape[0], -1, self.num_heads, self.head_dim)
        v = v.transpose(1, 2)

        if kv_cache is not None:
            K_full, V_full = kv_cache.update(layer_idx, k[0], v[0])
            # 补上batch维度
            K = K_full[torch.newaxis, :, :, :]
            V = V_full[torch.newaxis, :, :, :]
            # 如果输入序列长度为1，则更新缓存，用来区分是在解码还是预填充。。。靠
            if x.shape[1] == 1:
                kv_cache.advance(1)

        attn_out = F.scaled_dot_product_attention(q, K, V, attn_mask=None)
        attn_out = attn_out.transpose(1, 2).contiguous().view(
            attn_out.shape[0], -1, self.hidden_dim
        )
        return self.out_proj(attn_out)
    